In [1]:
!pip install groq faster-whisper gtts

INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.

In [2]:
from groq import Groq
from google.colab import userdata
userdata.get('HF_TOKEN')

client = Groq(
    api_key="HF_TOKEN"
)

SYSTEM_PROMPT = """
You are a multilingual Macau AI tutor.

Rules:
- Detect language automatically
- Reply in the same language
- Correct grammar politely
- Keep answers short
- Teach naturally
- Ask follow-up questions
- Encourage the learner
"""

In [ ]:
def ai_tutor(user_text):

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_text
            }
        ],

        temperature=0.7,
        max_tokens=300
    )

    return response.choices[0].message.content

In [ ]:
!pip install gtts langdetect

In [ ]:
from gtts import gTTS
from langdetect import detect

def text_to_speech(text):

    try:
        language = detect(text)
    except:
        language = "en"

    output_file = "reply.mp3"

    tts = gTTS(
        text=text,
        lang=language,
        slow=False
    )

    tts.save(output_file)

    return output_file

In [ ]:
from IPython.display import Audio

def voice_to_voice(audio_path):

    # Voice -> Text
    user_text = speech_to_text(audio_path)

    print("\n USER:")
    print(user_text)

    # Text -> AI
    ai_reply = ai_tutor(user_text)

    print("\n AI:")
    print(ai_reply)

    # AI Text -> Voice
    audio_file = text_to_speech(ai_reply)

    return Audio(audio_file, autoplay=True)

In [ ]:
!pip install -q faster-whisper

In [ ]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    "medium",
    device="cuda",
    compute_type="float16"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def speech_to_text(audio_path):

    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5
    )

    text = ""

    for segment in segments:
        text += segment.text + " "

    print("Language:", info.language)

    return text.strip()

In [ ]:
from google.colab import output
from base64 import b64decode

In [ ]:
RECORD_JS = """
async function recordAudio() {

    const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
    });

    const recorder = new MediaRecorder(stream);

    let chunks = [];

    recorder.ondataavailable = e => {
        chunks.push(e.data);
    };

    recorder.start();

    alert("Recording started. Speak now for 120 seconds.");

    await new Promise(resolve => setTimeout(resolve, 5000));

    recorder.stop();

    await new Promise(resolve => {
        recorder.onstop = resolve;
    });

    stream.getTracks().forEach(track => track.stop());

    const blob = new Blob(chunks, {
        type: "audio/webm"
    });

    const reader = new FileReader();

    reader.readAsDataURL(blob);

    await new Promise(resolve => {
        reader.onloadend = resolve;
    });

    return reader.result;
}

recordAudio();
"""

In [ ]:
output.eval_js("""
navigator.mediaDevices.getUserMedia({audio:true})
.then(() => "MIC OK")
.catch(err => err.toString())
""")

In [ ]:
audio_data = output.eval_js(RECORD_JS)

In [ ]:
audio_bytes = b64decode(
    audio_data.split(",")[1]
)

with open("input.webm", "wb") as f:
    f.write(audio_bytes)

print("Saved as input.webm")

In [ ]:
import os

print(os.path.exists("input.webm"))

In [ ]:
user_text = speech_to_text("input.webm")

print(user_text)

In [ ]:
voice_to_voice("input.webm")

In [ ]:
def speech_to_text(audio_path):

    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5
    )

    text = " ".join(
        segment.text
        for segment in segments
    )

    print("Language:", info.language)

    return text